# Automation and workflowing

A rule says which messages. A workflow says what to do: a Python function made of
steps, with logs and a JSON artifact coming out. Run it from cron, a timer, or an Ansible play.


In [1]:
from amoorion import Amoorion

from amoorion.models import Message

# A sandbox, with its own inbox.
amo = Amoorion()

# What hooks would have delivered.
amo.store.keep(
    [
        Message(
            platform="hook:generic",
            text="deploy finished",
            id="1",
        ),
        Message(
            platform="hook:generic",
            text="backup finished",
            id="2",
        ),
    ]
)

# A rule: which messages, and what steps. `$text` is the text of the message.
rule = {
    "name": "announce-deploys",
    "platform": "hook:generic",
    "contains": "deploy",
    "steps": [
        {
            "method": "publish",
            "params": {
                "text": "Status: $text",
                "to": ["dryrun"],
            },
        }
    ],
}

announce-deploys 1 True Status: deploy finished


## A workflow, step by step

Decorated, it is kept under its name. Rules say when it runs by itself; its own steps run one after another, stopping at the first that fails.


In [ ]:
# `step` logs a step, `EXECUTE` runs them, `log` adds a line of your own.
from amoorion.workflows import EXECUTE, log, step


@amo.workflow("announce", rules=[rule])
def announce(
    amo,
    inputs,
):
    """Write a short status, then post it."""

    @step
    def write():
        """What this step answers lands in the artifact, under its name."""

        log(f"got: {inputs.get('text')}")

        return {"status": f"Status: {inputs.get('text')}"}

    @step
    def send():
        """A step fails by raising or by answering False."""

        return amo.publish(text="the status", to=["dryrun"])[0]["ok"]

    return EXECUTE(
        write,
        send,
    ).serial()


# Run by name, with inputs of your own.
run = amo.perform(
    "announce",
    {"text": "by hand"},
)

run.ok, run.artifact, run.logs

## At the same time

The same with async steps; `.paralel()` runs them together. Plain functions run in threads next to them.


In [ ]:
import asyncio


@amo.workflow("gather")
async def gather(
    amo,
    inputs,
):
    """Two lookups at once."""

    @step
    async def first():

        await asyncio.sleep(0.01)

        return "from first"

    @step
    def second():

        return "from second"

    return await EXECUTE(
        first,
        second,
    ).paralel()


amo.perform("gather").artifact

## Reacting to what arrives

`react` runs workflows on the new messages their rules match, once per message:


In [ ]:
for done in amo.react():
    print(
        done.name,
        done.ok,
        done.artifact,
    )

# The second run finds nothing new: this is safe to run from a timer.
amo.react()

A rule may have steps of its own (methods, as plain data), which run first. A step that fails leaves the message unhandled, so the next run tries again:


In [ ]:
amo.store.keep(
    [
        Message(
            platform="hook:generic",
            text="deploy two",
            id="3",
        )
    ]
)

# A rule that can't succeed: this platform doesn't exist.
broken = rule | {
    "name": "to-nowhere",
    "contains": "two",
    "steps": [
        {
            "method": "publish",
            "params": {"text": "$text", "to": ["nowhere"]},
        }
    ],
}

# Without Python: the method `workflow` runs rules by themselves.
print([done["ok"] for done in amo.workflow(rules=[broken])])

# Tried again, as failures aren't marked as handled.
print([done["ok"] for done in amo.workflow(rules=[broken])])

## Rules in files, logs and artifacts in files

A rule is plain data, so it lives in a YAML file; the result of a run is a JSON file, ready for whatever posts it.


In [ ]:
from pathlib import Path

import json

import tempfile

import yaml

folder = Path(tempfile.mkdtemp())

# A rule in a file.
(folder / "deploys.rule.yml").write_text(
    yaml.safe_dump(
        {"rules": [rule]},
        sort_keys=False,
    )
)

print((folder / "deploys.rule.yml").read_text())


# Rules can be a file, a directory of `*.rule.yml`, or a list.
@amo.workflow("from-file", rules=folder)
def from_file(
    amo,
    inputs,
):

    return {"link": "https://example.org/" + inputs["id"]}


# Its artifact, for what comes next.
saved = amo.perform(
    "from-file",
    {"id": "42"},
).save(folder / "run.json")

json.loads(saved.read_text())